## Example Set: Agents - Intermediate v2

In [ ]:
import os

from crewai import Agent, Crew, Process, Task
from crewai.llm import LLM
from dotenv import load_dotenv

load_dotenv()

llm = LLM(
    model="openai/gpt-4o",
    api_key=os.getenv("OPENROUTER_API_KEY"),
    base_url="https://openrouter.ai/api/v1",
)

---

### Structured Output (`output_pydantic`)

Get typed Python objects instead of raw text. Define a Pydantic model and CrewAI auto-parses the agent's output into it.

In [ ]:
from pydantic import BaseModel, Field


class ProsCons(BaseModel):
    topic: str = Field(description="The topic being analyzed")
    pros: list[str] = Field(description="List of advantages")
    cons: list[str] = Field(description="List of disadvantages")
    verdict: str = Field(description="One-line recommendation")


analyst = Agent(
    role="Technology Analyst",
    goal="Provide balanced analysis of technology topics",
    llm=llm,
    backstory="You are a technology analyst known for objective, balanced assessments.",
)

analysis_task = Task(
    description="Analyze the pros and cons of {topic}. Include at least 3 pros and 3 cons.",
    expected_output="Structured pros and cons analysis",
    output_pydantic=ProsCons,
    agent=analyst,
)

---

### Code Guardrail

A Python function that validates the agent's output. Returns `(True, data)` to pass or `(False, "reason")` to make the agent retry.

In [ ]:
from crewai.tasks.task_output import TaskOutput


def validate_analysis(result: TaskOutput) -> tuple[bool, any]:
    report = result.pydantic
    if not report:
        return (False, "Output must be structured")
    if len(report.pros) < 3:
        return (False, f"Only {len(report.pros)} pros found. Need at least 3.")
    if len(report.cons) < 3:
        return (False, f"Only {len(report.cons)} cons found. Need at least 3.")
    return (True, report)


guarded_task = Task(
    description="Analyze the pros and cons of {topic}.",
    expected_output="Structured analysis with at least 3 pros and 3 cons",
    output_pydantic=ProsCons,
    guardrail=validate_analysis,
    agent=analyst,
)

---

### No-Code Guardrail

Pass a plain English string as the guardrail. CrewAI uses an LLM to evaluate whether the output meets the criteria.

In [ ]:
writer = Agent(
    role="Technical Writer",
    goal="Write clear, specific technical summaries",
    llm=llm,
    backstory="You are a technical writer who always includes concrete data.",
)

summary_task = Task(
    description="Write a 2-paragraph summary about {topic}.",
    expected_output="A concise, data-driven summary",
    guardrail="The summary must include at least 2 specific numbers or statistics. "
    "Reject if it only uses vague phrases like 'significant growth' without figures.",
    agent=writer,
)

---

---

### Memory (`memory=True`)

Enable on the Crew. Agents remember context from previous runs and produce richer answers over time.

In [ ]:
knowledge_agent = Agent(
    role="Knowledge Specialist",
    goal="Provide thorough answers that build on prior context",
    llm=llm,
    backstory="You are a knowledgeable analyst who gives richer answers over time.",
)

qa_task = Task(
    description="Answer: {question}",
    expected_output="A detailed, well-structured answer",
    agent=knowledge_agent,
)

memory_crew = Crew(
    agents=[knowledge_agent],
    tasks=[qa_task],
    process=Process.sequential,
    memory=True,
)

---